In [1]:
%load_ext autoreload
%autoreload 2

# Walkthrough Example

This document provides a walkthrough of how to use the `oceanicospy` package to handle observations from various sensors.

# Importing libraries

All the sensors are represented as objects in the module `oceanicospy.observations`.

In [2]:
from oceanicospy.observations import pressure_sensors,weather_stations,AWAC,Buoy,ctd,HOBO_Temp,HOBO_TempCond
from datetime import datetime, timedelta
import glob

# Pressure sensors: AQUALogger, RBR, BlueLogger

## Data directories and sampling configuration

For every pressure sensor, we specify the directory where the data files are located and a sampling configuration dictionary that contains parameters such as anchoring depth, sensor height, sampling frequency, burst length, and optional start and end times for filtering the data.

In [3]:
measurement_pressure_sensors_paths = ['../data/observations/AQUAlogger/','../data/observations/RBR/','../data/observations/Bluelog/']

sampling_AQ = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=1,burst_length_s=2048,
                    start_time=datetime(2025,5,9,10,0,0),end_time=datetime(2025,5,19,18,0,0)-timedelta(seconds=1))
sampling_RBR = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=2, burst_length_s=7200,
                    start_time=datetime(2025,5,9,10,0,0),end_time=datetime(2025,5,19,18,0,0)-timedelta(seconds=0.5))
sampling_Bluelog = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=2, burst_length_s=4096, temperature=False,
                        start_time=datetime(2025,7,4,14,0,0), end_time=datetime(2025,7,10,12,0,0)-timedelta(seconds=1))

## Reading and storing data

In [4]:

sampling_data = [sampling_AQ,sampling_RBR,sampling_Bluelog]
metadata_list=['AQ','RBR','Bluelog']
dict_raw_measurements = dict()
dict_clean_measurements = dict() 

for idx,measurement_path in enumerate(measurement_pressure_sensors_paths):
    print(measurement_path)
    if 'AQ' in measurement_path:
        object_device = pressure_sensors.AQUAlogger(measurement_path, sampling_data[idx],filename='AQUAlogger_520PT5.csv')
    elif 'Bluelog' in measurement_path:
        object_device = pressure_sensors.Bluelog(measurement_path, sampling_data[idx])
    else:
        object_device = pressure_sensors.RBR(measurement_path,sampling_data[idx])
    raw_data = object_device.get_raw_records()
    clean_data = object_device.get_clean_records()
    dict_raw_measurements[metadata_list[idx]] = raw_data
    dict_clean_measurements[metadata_list[idx]] = clean_data

../data/observations/AQUAlogger/
../data/observations/RBR/
../data/observations/Bluelog/


A quick look at the data dictionary can show the available data for each sensor. Each sensor's data is retrieved as a pandas DataFrame, which allows for easy manipulation and analysis. The raw records contain the original columns from the sensor files, which may have different names and formats. The clean records are then obtained by applying the `get_clean_records` method of the `PressureSensor` class, which performs several processing steps to prepare the data for analysis. This includes standardizing column names, parsing dates, computing depth from pressure, and assigning burst IDs.

In [5]:
dict_clean_measurements['AQ'].head()

,pressure[bar],depth[m],depth_aux[m],burstId,eta[m]
date,,,,,
2025-05-09 10:00:00,1.245722,3.136509,2.307327,1,0.052596
2025-05-09 10:00:01,1.244709,3.124966,2.297273,1,0.041052
2025-05-09 10:00:02,1.237181,3.039040,2.222556,1,-0.044874
2025-05-09 10:00:03,1.243695,3.113418,2.287209,1,0.029503
2025-05-09 10:00:04,1.253249,3.222086,2.382034,1,0.138170


In [6]:
dict_clean_measurements['RBR'].head()

,pressure[bar],depth[m],burstId,eta[m]
date,,,,
2025-05-09 10:00:00.000,1.241829,2.267149,1,0.117985
2025-05-09 10:00:00.500,1.239702,2.246051,1,0.096886
2025-05-09 10:00:01.000,1.235314,2.202530,1,0.053365
2025-05-09 10:00:01.500,1.233978,2.189276,1,0.040110
2025-05-09 10:00:02.000,1.234466,2.194119,1,0.044954


In [7]:
dict_clean_measurements['Bluelog'].head()

,pressure[bar],temperature[c],depth[m],burstId,eta[m]
date,,,,,
2025-07-04 14:00:00.000,1.0615,27.8,0.478890,1,-0.000014
2025-07-04 14:00:00.500,1.0616,27.8,0.479883,1,0.000980
2025-07-04 14:00:01.000,1.0617,27.8,0.480875,1,0.001974
2025-07-04 14:00:01.500,1.0617,27.8,0.480875,1,0.001976
2025-07-04 14:00:02.000,1.0615,27.8,0.478890,1,-0.000008


The clean records for every sensor derived from the same abstract class share the same column names and formats, which allows for easy comparison and combination of data from different sensors.

# Weather stations: Davis Vantage Pro, WeatherSens and Rainwise

Currently, there is support for three different weather station models: Davis Vantage Pro, WeatherSens and Rainwise

In [8]:
Davis_weather_station = weather_stations.DavisVantagePro('../data/observations/WeatherStations/DavisVantagePro.txt')
WeatherSens_weather_station = weather_stations.WeatherSens('../data/observations/WeatherStations/WeatherSens.xlsx')
Rainwise_weather_station = weather_stations.Rainwise('../data/observations/WeatherStations/Rainwise.csv')

object_list = [Davis_weather_station,WeatherSens_weather_station,Rainwise_weather_station]

In [9]:
dict_raw_measurements = dict()
dict_clean_measurements = dict()

for ws in object_list:
    raw_data = ws.get_raw_records()
    clean_data = ws.get_clean_records()
    dict_raw_measurements[ws.__class__.__name__] = raw_data
    dict_clean_measurements[ws.__class__.__name__] = clean_data

The `get_raw_records` method of the `WeatherStation` class reads the data files for each weather station and stores the raw records in a dictionary. The raw records contain the original columns from the sensor files, which may have different names and formats. An example of the raw records for the Davis Vantage Pro weather station is shown below.

In [10]:
dict_raw_measurements['DavisVantagePro'].head()

,Date,time,AM/PM,Out,Temp1,Temp2,Hum,Pt.,Speed,Dir1,...,Temp4,Hum2,Dew,Heat,EMC,Density,Samp,Tx,Recept,Int.
0,08/19/23,12:15,AM,---,---,---,---,---,0.0,---,...,28.8,79,24.8,34.2,15.47,1.1319,0,1,0.0,15
1,08/19/23,12:30,AM,---,---,---,---,---,0.0,---,...,28.7,78,24.4,33.7,15.18,1.1329,0,1,0.0,15
2,08/19/23,12:45,AM,---,---,---,---,---,0.0,---,...,28.4,78,24.2,33.1,15.19,1.1339,0,1,0.0,15
3,08/19/23,1:00,AM,---,---,---,---,---,0.0,---,...,28.6,80,24.8,33.8,15.81,1.1321,0,1,0.0,15
4,08/19/23,1:15,AM,---,---,---,---,---,0.0,---,...,28.7,81,25.1,34.3,16.21,1.1307,0,1,0.0,15


A quick look to the column names for every weather station model shows that they are not under the same standard.

In [11]:
for ws in object_list:
    print(ws.__class__.__name__)
    print(dict_raw_measurements[ws.__class__.__name__].info())


DavisVantagePro
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 31 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Date     1218 non-null   object
 1   time     1218 non-null   object
 2   AM/PM    1218 non-null   object
 3   Out      1218 non-null   object
 4   Temp1    1218 non-null   object
 5   Temp2    1218 non-null   object
 6   Hum      1218 non-null   object
 7   Pt.      1218 non-null   object
 8   Speed    1218 non-null   object
 9   Dir1     1218 non-null   object
 10  Run      1218 non-null   object
 11  Speed2   1218 non-null   object
 12  Dir2     1218 non-null   object
 13  Chill    1218 non-null   object
 14  Index1   1218 non-null   object
 15  Index2   1218 non-null   object
 16  Bar      1218 non-null   object
 17  Rain     1218 non-null   object
 18  Rate     1218 non-null   object
 19  D-D1     1218 non-null   object
 20  D-D2     1218 non-null   object
 21  Temp4    1218 non-nul

For the `get_clean_records` method, all the column names are standardize in terms of names and dtypes. Additionally, non standard ways to define non-recorded data are replaced with `NaN` values in the clean records. This keep consistency in the data structure.

In [12]:
dict_clean_measurements['DavisVantagePro'].head()

,Temp1,Temp2,Speed,Dir1,Run,Speed2,Dir2,Bar,Rain,Rate,Temp4,Hum2,Dew,Heat,EMC,Density,Samp,Tx,Recept,Int.
date,,,,,,,,,,,,,,,,,,,,
2023-08-19 00:15:00,NaN,NaN,0.0,NaN,0.0,0.0,NaN,1011.7,0.0,0.0,28.8,79,24.8,34.2,15.47,1.1319,0,1,0.0,15
2023-08-19 00:30:00,NaN,NaN,0.0,NaN,0.0,0.0,NaN,1011.6,0.0,0.0,28.7,78,24.4,33.7,15.18,1.1329,0,1,0.0,15
2023-08-19 00:45:00,NaN,NaN,0.0,NaN,0.0,0.0,NaN,1011.3,0.0,0.0,28.4,78,24.2,33.1,15.19,1.1339,0,1,0.0,15
2023-08-19 01:00:00,NaN,NaN,0.0,NaN,0.0,0.0,NaN,1011.1,0.0,0.0,28.6,80,24.8,33.8,15.81,1.1321,0,1,0.0,15
2023-08-19 01:15:00,NaN,NaN,0.0,NaN,0.0,0.0,NaN,1010.8,0.0,0.0,28.7,81,25.1,34.3,16.21,1.1307,0,1,0.0,15


In [13]:
dict_clean_measurements['WeatherSens'].head()

,Device Restarts,rain[mm],wind_speed[m/s],wind_direction[°],temp[C],air_humidity[%],pressure[hPa],solar_radiation[W/m2],battery[V],panel_supply[V],DBA (dBm)
date,,,,,,,,,,,
1999-12-31 19:15:00,NaN,0.0,3.2,80.400002,30.500000,82.199997,1010.099976,762.700012,11.53,0.07,-51.0
1999-12-31 19:30:00,NaN,0.0,2.6,87.500000,30.600000,82.199997,1010.200012,776.000000,11.52,0.05,-51.0
1999-12-31 19:45:00,NaN,0.0,2.0,75.400002,30.799999,81.800003,1010.299988,812.099976,11.51,0.06,-51.0
1999-12-31 20:00:00,NaN,0.0,2.4,73.900002,31.200001,80.599998,1010.200012,474.600006,11.51,0.07,-51.0
1999-12-31 20:15:00,NaN,0.0,2.1,66.300003,31.200001,80.699997,1010.200012,742.599976,11.49,0.05,-51.0


In [14]:
dict_clean_measurements['Rainwise'].head()

,Temp Avg,Temp Low,Temp High,Heat Index,Wind Chill,Temp (Day) Low,Temp (Day) High,Hum Avg,Hum Low,Hum High,...,Inside Temp (Day) High,Station Voltage,UV Index,Temp 1 Avg,Temp 1 Low,Temp 1 High,Solar Radiation 2 Avg,Solar Radiation 2 Sum (Interval),Solar Radiation 2 Sum (Day),Battery Voltage 2
date,,,,,,,,,,,,,,,,,,,,,
2023-11-09 11:59:14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-11-09 12:00:00,31.4,NaN,NaN,45.7,31.4,NaN,NaN,88.0,NaN,NaN,...,NaN,6.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.9
2023-11-09 12:15:00,33.9,NaN,NaN,52.6,33.9,NaN,NaN,81.0,NaN,NaN,...,NaN,6.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.9
2023-11-09 12:31:07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-11-09 12:45:00,34.8,NaN,NaN,39.2,34.8,NaN,NaN,47.0,NaN,NaN,...,NaN,6.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.9


In [15]:
for ws in object_list:
    print(ws.__class__.__name__)
    print(dict_clean_measurements[ws.__class__.__name__].info())

DavisVantagePro
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1218 entries, 2023-08-19 00:15:00 to 2023-08-31 16:30:00
Data columns (total 20 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Temp1    2 non-null      float64
 1   Temp2    2 non-null      float64
 2   Speed    1218 non-null   float64
 3   Dir1     0 non-null      float64
 4   Run      1218 non-null   float64
 5   Speed2   1218 non-null   float64
 6   Dir2     0 non-null      float64
 7   Bar      1218 non-null   float64
 8   Rain     1218 non-null   float64
 9   Rate     1218 non-null   float64
 10  Temp4    1218 non-null   float64
 11  Hum2     1218 non-null   int64  
 12  Dew      1218 non-null   float64
 13  Heat     1218 non-null   float64
 14  EMC      1218 non-null   float64
 15  Density  1218 non-null   float64
 16  Samp     1218 non-null   int64  
 17  Tx       1218 non-null   int64  
 18  Recept   1218 non-null   float64
 19  Int.     1218 non-null   int64  
dtype

# CTD sensors

In [16]:
sea_sun_marine_tech = ctd.SeaSunMarineTechCTD('../data/observations/CTD/SeaSunMarineTech.TOB')
castaway = ctd.CastawayCTD('../data/observations/ctd/CC2349006_20240301_185500.csv',has_header=False)

When the file recorded by the Castaway CTD sensor does not have a header, we can specify that in the constructor of the `CastawayCTD` class by setting the `has_header` parameter to `False`.

If header is not in the file, the class will look for metadata information in the summary file that is generated by the Castaway software when the data is downloaded from the sensor. This summary file contains information about the cast, such as the date and time of the cast, the location, and the sensor model. 

In [17]:
ctd_objects = [sea_sun_marine_tech, castaway]

In [18]:
dict_ctd_raw_measurements = dict()
dict_ctd_clean_measurements = dict()

for ctd_obj in ctd_objects:
    print(ctd_obj.cast_time)
    raw_data = ctd_obj.get_raw_records()
    clean_data = ctd_obj.get_clean_records()
    dict_ctd_raw_measurements[ctd_obj.__class__.__name__] = raw_data
    dict_ctd_clean_measurements[ctd_obj.__class__.__name__] = clean_data

2025-01-26 18:12:50
2024-03-01 18:54:59.999996298


In [19]:
dict_ctd_raw_measurements['SeaSunMarineTechCTD'].head()

,Datasets,Press,Temp,Cond,Turb,SALIN,IntD,IntT
0,1,-1.23,26.60,0.3,0.75,0.14,45683.46,45683.46
1,2,1.36,26.61,0.3,0.75,0.14,45683.46,45683.46
2,3,1.36,26.63,0.3,0.76,0.14,45683.46,45683.46
3,4,1.36,26.64,0.3,0.75,0.14,45683.46,45683.46
4,5,1.36,26.64,0.3,0.75,0.14,45683.46,45683.46


In [20]:
dict_ctd_raw_measurements['CastawayCTD'].head()

,Pressure (Decibar),Depth (Meter),Temperature (Celsius),Conductivity (MicroSiemens per Centimeter),Specific conductance (MicroSiemens per Centimeter),Salinity (Practical Salinity Scale),Sound velocity (Meters per Second),Density (Kilograms per Cubic Meter)
0,0.15,0.149818,28.007169,58217.828125,54915.051157,36.394542,1542.819547,1023.444292
1,0.45,0.449452,28.003554,58216.687377,54917.720512,36.396442,1542.818669,1023.448193
2,0.75,0.749086,28.001095,58212.049293,54915.892851,36.394978,1542.816775,1023.449183
3,1.05,1.048720,27.996325,58201.985534,54911.341388,36.391474,1542.807647,1023.449396
4,1.35,1.348354,27.996636,58200.045029,54909.187730,36.389770,1542.811585,1023.449295


Similar to the previous cases with other sensors, when we use the raw records, we can see that the column names and formats are different for each CTD sensor model.

In [21]:
for key in dict_ctd_raw_measurements.keys():
    print(f"Raw records for {key}:")
    print(dict_ctd_raw_measurements[key].columns)
    print("\n")

Raw records for SeaSunMarineTechCTD:
Index(['Datasets', 'Press', 'Temp', 'Cond', 'Turb', 'SALIN', 'IntD', 'IntT'], dtype='object')


Raw records for CastawayCTD:
Index(['Pressure (Decibar)', 'Depth (Meter)', 'Temperature (Celsius)',
       'Conductivity (MicroSiemens per Centimeter)',
       'Specific conductance (MicroSiemens per Centimeter)',
       'Salinity (Practical Salinity Scale)',
       'Sound velocity (Meters per Second)',
       'Density (Kilograms per Cubic Meter)'],
      dtype='object')




Because of the different formats and column names in the raw records, it is not straightforward to compare or combine data from different CTD sensors. However, by using the `get_clean_records` method of each CTD sensor class, we can standardize the column names and formats, which allows for easy comparison and combination of data from different CTD sensors.

In [22]:
dict_ctd_clean_measurements['SeaSunMarineTechCTD'].head()

,temperature[C],conductivity[mS/cm],turbidity[FTU],salinity[PSU],date_serial,time_serial
pressure[dbar],,,,,,
-1.23,26.60,0.3,0.75,0.14,45683.46,45683.46
1.36,26.61,0.3,0.75,0.14,45683.46,45683.46
1.36,26.63,0.3,0.76,0.14,45683.46,45683.46
1.36,26.64,0.3,0.75,0.14,45683.46,45683.46
1.36,26.64,0.3,0.75,0.14,45683.46,45683.46


In [23]:
dict_ctd_clean_measurements['CastawayCTD'].head()

,pressure[dbar],temperature[C],conductivity[uS/cm],specific_conductance[uS/cm],salinity[PSS],sound_velocity[m/s],density[kg/m3]
depth[m],,,,,,,
0.149818,0.15,28.007169,58217.828125,54915.051157,36.394542,1542.819547,1023.444292
0.449452,0.45,28.003554,58216.687377,54917.720512,36.396442,1542.818669,1023.448193
0.749086,0.75,28.001095,58212.049293,54915.892851,36.394978,1542.816775,1023.449183
1.048720,1.05,27.996325,58201.985534,54911.341388,36.391474,1542.807647,1023.449396
1.348354,1.35,27.996636,58200.045029,54909.187730,36.389770,1542.811585,1023.449295


If the header is present in the file, the class will look for metadata information in the header of the file. This header contains information about the cast, such as the date and time of the cast, the location, etc.

In [24]:
ctd_castaway_header = ctd.CastawayCTD('../data/observations/ctd/CastAway_with_header.csv',has_header=True)
ctd_castaway_header_raw = ctd_castaway_header.get_raw_records()
ctd_castaway_header_clean = ctd_castaway_header.get_clean_records()

In [25]:
ctd_castaway_header_clean

,pressure[dbar],temperature[C],conductivity[uS/cm],specific_conductance[uS/cm],salinity[PSS],sound_velocity[m/s],density[kg/m3]
depth[m],,,,,,,
0.149915,0.150000,29.131853,58896.679687,54401.130407,36.009578,1544.844117,1022.781047
0.449744,0.450000,29.129037,58900.812264,54407.778109,36.014463,1544.848315,1022.786945
0.749572,0.750000,29.125827,58886.989676,54398.235556,36.007252,1544.838995,1022.783889
1.049400,1.050000,29.128301,58906.405764,54413.684773,36.018701,1544.861316,1022.792939
1.349227,1.350000,29.123365,58885.300862,54399.150344,36.007763,1544.844420,1022.787665
1.649055,1.650000,29.123191,58884.665700,54398.737991,36.007366,1544.848696,1022.788706
1.948882,1.950000,29.117810,58884.388325,54403.890349,36.011141,1544.846277,1022.794632
2.248708,2.250000,29.113707,58880.493284,54404.416588,36.011455,1544.842944,1022.797529
2.548532,2.550000,29.106853,58883.008041,54413.632366,36.018269,1544.840576,1022.806233


# ADCP sensors: AWAC

Support for ADCP sensors, such as the AWAC, is also included in the module. The `AWAC` class is designed to handle the specific data formats and requirements of ADCP measurements, allowing users to easily load and process wave records from AWAC deployments.

Similar to the pressure sensors, the `AWAC` class also requires a directory path and sampling configuration for initialization. 

In [26]:
measurement_AWAC_paths = ['../data/observations/AWAC/']

sampling_AWAC = dict(
    anchoring_depth=1,
    sensor_height=0.2,
    sampling_freq=1,
    burst_length_s=2048,
    temperature=False,
    start_time=datetime(2023,8,18,19,0,0),
    end_time=datetime(2023,9,1,15,0,0) - timedelta(seconds=1))

In [27]:
help(AWAC)

Help on class AWAC in module oceanicospy.observations.awac:

class AWAC(builtins.object)
 |  AWAC(directory_path, sampling_data)
 |  
 |  Handle reading and processing data files recorded by an ADCP AWAC (Nortek S.A.).
 |  
 |  Parameters
 |  ----------
 |  directory_path : str
 |      Path to the directory containing the ``.hdr`` and ``.wad`` files.
 |      Must end with a path separator (e.g. ``'data/awac/'``).
 |  sampling_data : dict
 |      Dictionary containing information about the device installation.
 |      Must include the keys ``'start_time'`` and ``'end_time'``, both
 |      parseable by :func:`pandas.to_datetime`.
 |  
 |  Notes
 |  -----
 |  - 04-Jan-2018 : Origination - Daniel Peláez
 |  - 01-Sep-2023 : Migration to Python - Alejandro Henao
 |  - 10-Dec-2024 : Class implementation - Franklin Ayala
 |  
 |  Methods defined here:
 |  
 |  __init__(self, directory_path, sampling_data)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  get_clean

In [28]:
AWAC_obj = AWAC(measurement_AWAC_paths[0],sampling_AWAC)

## Reading wave records

The method `get_raw_wave_records` can be used to retrieve the raw wave records, with an option to specify whether to read from a single WAD file or multiple files in the directory.

### Using the.wad file with all the records

In [29]:
df_raw_wave_single_wad = AWAC_obj.get_raw_wave_records(from_single_wad=True)
df_clean_wave_single_wad = AWAC_obj.get_clean_wave_records(from_single_wad=True)

/Users/franklinayala/Documents/projects/oceanicospy/oceanicospy/observations/awac.py:298: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["pressure[bar]"] = df["pressure[dbar]"] / 10.0


In [30]:
df_raw_wave_single_wad.head()

,month,day,year,hour,minute,second,3 Pressure (dbar),4 Spare,5 Analog input,6 Velocity (Beam1) (m/s),7 Velocity (Beam2) (m/s),8 Velocity (Beam3) (m/s),9 Velocity (Beam4) (m/s),10 Amplitude (Beam1) (counts),11 Amplitude (Beam2) (counts),12 Amplitude (Beam3) (counts),13 Amplitude (Beam4) (counts)
0,8,18,2023,19,0,1.0,0.0,13.286,14.561,13,0,-1.780,-0.601,-3.245,24,24,20
1,8,18,2023,19,0,1.5,0.0,14.707,12.990,14,0,-0.173,1.524,-4.112,22,23,18
2,8,18,2023,19,0,2.0,0.0,13.975,15.950,12,0,2.813,2.377,-3.706,24,24,19
3,8,18,2023,19,0,2.5,0.0,14.523,13.978,14,0,-2.034,-2.001,2.567,23,24,19
4,8,18,2023,19,0,3.0,0.0,10.388,16.318,14,0,-1.913,0.129,2.931,23,23,19


In [31]:
df_clean_wave_single_wad.head()

,pressure[bar],velocitybeam1[m/s],velocitybeam2[m/s],velocitybeam3[m/s],velocitybeam4[m/s],burstId
date,,,,,,
2023-08-18 19:00:01.000,0.0,13,0,-1.780,-0.601,1
2023-08-18 19:00:01.500,0.0,14,0,-0.173,1.524,1
2023-08-18 19:00:02.000,0.0,12,0,2.813,2.377,1
2023-08-18 19:00:02.500,0.0,14,0,-2.034,-2.001,1
2023-08-18 19:00:03.000,0.0,14,0,-1.913,0.129,1


### Using .wad files recorded per burst

In [32]:
df_raw_wave_combined_wad = AWAC_obj.get_raw_wave_records(from_single_wad=False)
df_clean_wave_combined_wad = AWAC_obj.get_clean_wave_records(from_single_wad=False)

/Users/franklinayala/Documents/projects/oceanicospy/oceanicospy/observations/awac.py:298: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["pressure[bar]"] = df["pressure[dbar]"] / 10.0


In [33]:
df_raw_wave_combined_wad.head()

,burstId,2 Ensemble counter,3 Pressure (dbar),4 Spare,5 Analog input,6 Velocity (Beam1) (m/s),7 Velocity (Beam2) (m/s),8 Velocity (Beam3) (m/s),9 Velocity (Beam4) (m/s),10 Amplitude (Beam1) (counts),11 Amplitude (Beam2) (counts),12 Amplitude (Beam3) (counts),13 Amplitude (Beam4) (counts)
0,1,1,0.0,13.286,14.561,13,0,-1.780,-0.601,-3.245,24,24,20
1,1,2,0.0,14.707,12.990,14,0,-0.173,1.524,-4.112,22,23,18
2,1,3,0.0,13.975,15.950,12,0,2.813,2.377,-3.706,24,24,19
3,1,4,0.0,14.523,13.978,14,0,-2.034,-2.001,2.567,23,24,19
4,1,5,0.0,10.388,16.318,14,0,-1.913,0.129,2.931,23,23,19


In [34]:
df_clean_wave_combined_wad.head()

,pressure[bar],burstId,velocitybeam1[m/s],velocitybeam2[m/s],velocitybeam3[m/s],velocitybeam4[m/s]
2023-08-18 19:00:00.000,0.0,1,13,0,-1.780,-0.601
2023-08-18 19:00:00.500,0.0,1,14,0,-0.173,1.524
2023-08-18 19:00:01.000,0.0,1,12,0,2.813,2.377
2023-08-18 19:00:01.500,0.0,1,14,0,-2.034,-2.001
2023-08-18 19:00:02.000,0.0,1,14,0,-1.913,0.129


## Reading current records

In [35]:
x_currents_raw,y_currents_raw = AWAC_obj.get_raw_currents_records()

In [36]:
x_currents_raw.head()

,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,...,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,cell_23,cell_24,cell_25,cell_26
0,5.858,2.155,1.184,-0.752,-0.320,-0.186,-0.531,-0.454,-0.264,-0.505,...,-0.765,-0.718,-0.466,-0.651,-0.516,-0.760,-0.500,-0.583,-0.531,-0.792
1,5.066,1.388,1.375,-0.306,-0.914,-1.109,-0.326,-0.781,-0.969,-0.845,...,-0.787,-0.957,-0.992,-0.718,-0.924,-0.924,-1.162,-1.027,-0.815,-0.901
2,5.291,1.272,1.047,-0.908,-1.080,-0.526,-0.659,-0.788,-0.857,-0.782,...,-0.633,-0.848,-0.946,-1.212,-0.807,-1.084,-1.228,-0.706,-0.674,-0.703
3,5.837,1.354,0.851,-1.224,-0.305,-0.741,-0.887,-0.402,-1.112,-0.538,...,-0.980,-0.733,-0.872,-0.713,-0.859,-0.808,-1.001,-0.843,-0.967,-0.490
4,5.455,1.276,0.416,-0.563,-0.553,-0.836,-0.744,-1.075,-0.388,-1.072,...,-0.552,-1.110,-0.557,-1.027,-0.638,-0.314,-1.031,-0.830,-0.714,-1.002


In [37]:
y_currents_raw.head()

,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,...,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,cell_23,cell_24,cell_25,cell_26
0,0.581,0.683,0.058,0.720,0.583,0.744,0.745,0.482,1.127,0.846,...,0.436,0.984,0.659,0.216,0.487,0.727,0.896,0.307,0.940,0.801
1,0.131,0.243,0.087,0.867,0.647,0.483,0.850,0.916,0.368,0.921,...,0.300,0.861,0.946,0.366,1.261,1.377,0.966,0.890,0.688,1.216
2,0.219,0.181,-0.189,0.769,1.040,0.835,0.922,0.643,0.476,0.943,...,0.772,0.912,1.036,0.672,0.990,0.849,0.314,1.008,1.130,0.921
3,0.423,0.087,0.289,0.432,0.807,0.773,0.303,0.590,0.488,0.548,...,0.870,0.685,1.256,0.623,0.547,0.933,0.774,0.847,0.526,0.689
4,0.278,-0.048,-0.105,0.834,0.659,0.616,0.965,0.971,1.076,0.901,...,0.915,0.953,0.979,0.990,0.985,0.521,1.035,0.979,0.487,0.837


In [38]:
help(AWAC.get_clean_currents_records)

Help on function get_clean_currents_records in module oceanicospy.observations.awac:

get_clean_currents_records(self, compute_speed_dir=True)
    Processes the raw current data by reading the x and y components, setting the index to a date range, and optionally computing speed and direction.
    
    Parameters
    ----------
    compute_speed_dir : bool, optional
        If True, computes the current speed (magnitude) and direction from the x and y components. Default is True.
    
    Returns
    -------
    x_component_clean : pandas.DataFrame
        Cleaned DataFrame of the x component of the current.
    y_component_clean : pandas.DataFrame
        Cleaned DataFrame of the y component of the current.
    current_speed : pandas.DataFrame, optional
        DataFrame of the current speed for each depth cell, only returned if compute_speed_dir is True.
    current_dir : pandas.DataFrame, optional
        DataFrame of the current direction for each depth cell, only returned if comput

The method `get_clean_currents_records` allows users to retrieve cleaned current records, which can be useful for further analysis and interpretation of the data. If the `compute_speed_dir` parameter is set to `True`, the method will also compute the current speed and direction based on the u and v components of the currents.

In [39]:
x_currents_clean,y_currents_clean,current_speed,current_dir = AWAC_obj.get_clean_currents_records()

/Users/franklinayala/Documents/projects/oceanicospy/oceanicospy/utils/wave_props.py:84: RuntimeWarning: invalid value encountered in scalar divide
  theta = 90 + (np.arctan(abs(y/x))*(180/np.pi))


In [40]:
x_currents_clean.head()

,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,...,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,cell_23,cell_24,cell_25,cell_26
2023-08-18 19:25:00,5.066,1.388,1.375,-0.306,-0.914,-1.109,-0.326,-0.781,-0.969,-0.845,...,-0.787,-0.957,-0.992,-0.718,-0.924,-0.924,-1.162,-1.027,-0.815,-0.901
2023-08-18 19:55:00,5.291,1.272,1.047,-0.908,-1.080,-0.526,-0.659,-0.788,-0.857,-0.782,...,-0.633,-0.848,-0.946,-1.212,-0.807,-1.084,-1.228,-0.706,-0.674,-0.703
2023-08-18 20:25:00,5.837,1.354,0.851,-1.224,-0.305,-0.741,-0.887,-0.402,-1.112,-0.538,...,-0.980,-0.733,-0.872,-0.713,-0.859,-0.808,-1.001,-0.843,-0.967,-0.490
2023-08-18 20:55:00,5.455,1.276,0.416,-0.563,-0.553,-0.836,-0.744,-1.075,-0.388,-1.072,...,-0.552,-1.110,-0.557,-1.027,-0.638,-0.314,-1.031,-0.830,-0.714,-1.002
2023-08-18 21:25:00,5.852,1.275,0.659,-1.052,-0.946,-0.900,-0.785,-0.626,-0.889,-0.693,...,-0.942,-1.013,-0.991,-0.724,-0.940,-0.756,-0.713,-0.700,-0.407,-0.323


In [41]:
current_dir.head()

,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,cell_10,...,cell_17,cell_18,cell_19,cell_20,cell_21,cell_22,cell_23,cell_24,cell_25,cell_26
2023-08-18 18:55:00,84.335894,72.414647,87.195523,313.754636,331.238274,345.963757,324.520564,316.713465,346.816166,329.165894,...,299.680314,323.882713,324.734646,288.355702,313.343854,313.728682,330.836922,297.770691,330.538172,315.323702
2023-08-18 19:25:00,88.518738,80.069749,86.379566,340.559965,305.293817,293.534450,339.016712,319.548424,290.795423,317.464210,...,290.866567,311.977288,313.640294,297.010181,323.767836,326.137442,309.737605,310.912259,310.170113,323.463258
2023-08-18 19:55:00,87.629821,81.901431,100.232595,310.261807,313.919076,327.791528,324.444691,309.214096,299.048927,320.332159,...,320.650023,317.082565,317.599940,299.006416,320.814750,308.068414,284.343221,324.992723,329.185606,322.645478
2023-08-18 20:25:00,85.855093,86.323567,71.242489,289.440035,339.296266,316.210827,288.860213,325.731207,293.694216,315.527571,...,311.597230,313.061250,325.229008,311.146077,302.488453,319.106665,307.712156,315.135611,298.543996,324.580490
2023-08-18 20:55:00,87.082592,92.154311,104.165798,325.978290,319.998299,306.384352,322.368335,312.090109,340.170960,310.046550,...,328.898315,310.648022,330.362379,313.949080,327.068235,328.923170,315.110931,319.708592,304.296836,309.872998


# Spotter buoy

In [42]:
csv_files_spotter = glob.glob('../data/observations/Buoy/*.csv')
dict_raw_spotter_sources = dict()
dict_clean_spotter_sources = dict()
for csv_file in csv_files_spotter:
    if 'sofar' in csv_file:
        sampling_spotter = dict(start_time=datetime(2025,5,11,0,0,0),
                        end_time=datetime(2025,5,21,15,0,0))
    else:
        sampling_spotter = dict(start_time=datetime(2023,5,11,0,0,0),
                                end_time=datetime(2023,5,21,15,0,0))

    spotter_object = Buoy(csv_file,sampling_spotter)
    dict_raw_spotter_sources[csv_file.split('/')[-1].split('.')[0]] = spotter_object.get_raw_records()
    dict_clean_spotter_sources[csv_file.split('/')[-1].split('.')[0]] = spotter_object.get_clean_records()

In [43]:
dict_raw_spotter_sources['spotter_data_aqualink'].head()

,timestamp,wind_speed_gfs,wind_direction_gfs,temp_alert_noaa,temp_weekly_alert_noaa,dhw_noaa,satellite_temperature_noaa,sst_anomaly_noaa,top_temperature_spotter,bottom_temperature_spotter,significant_wave_height_spotter,wave_mean_period_spotter,wave_mean_direction_spotter,wind_speed_spotter,wind_direction_spotter,significant_wave_height_sofar_model,wave_mean_period_sofar_model,wave_mean_direction_sofar_model
0,2023-08-31T12:00:00.000Z,NaN,NaN,2.0,2.0,3.749791,30.039919,1.888951,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-08-30T12:00:00.000Z,NaN,NaN,2.0,2.0,3.519906,30.179922,2.039599,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-08-29T12:00:00.000Z,NaN,NaN,2.0,2.0,3.439877,29.979915,1.850237,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-08-28T12:00:00.000Z,NaN,NaN,2.0,2.0,3.399530,30.049994,1.930962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-08-27T12:00:00.000Z,NaN,NaN,2.0,2.0,3.359647,29.899956,1.791569,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
dict_raw_spotter_sources['spotter_data_sofar'].head()

,Battery Voltage (V),Power (W),Humidity (%rel),Epoch Time,Significant Wave Height (m),Peak Period (s),Mean Period (s),Peak Direction (deg),Peak Directional Spread (deg),Mean Direction (deg),...,Partition0 Mean Direction (deg),Partition0 Mean Directional Spread (deg),Partition1 Start Frequency (hz),Partition1 End Frequency (hz),Partition1 Significant Wave Height (m),Partition1 Mean Period (s),Partition1 Mean Direction (deg),Partition1 Mean Directional Spread (deg),Mean Barometric Pressure (hPa),Processing Source
0,4.04,-0.15,82.4,1748751000,0.598,7.314,3.303,112.73,70.858,66.793,...,-,-,-,-,-,-,-,-,1012.2,embedded
1,4.04,-0.15,82.4,1748749200,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,embedded
2,4.04,-0.15,80.8,1748743800,0.576,7.314,3.264,122.074,69.724,59.15,...,-,-,-,-,-,-,-,-,1010.9,embedded
3,4.04,-0.15,80.8,1748742000,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,embedded
4,4.04,-0.08,75.2,1748736600,0.582,8.533,3.373,111.922,73.695,55.895,...,-,-,-,-,-,-,-,-,1009.5,embedded


In [45]:
for key in dict_raw_spotter_sources.keys():
    print(f"Raw records for {key}:")
    print(dict_raw_spotter_sources[key].info())

Raw records for spotter_data_aqualink:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9356 entries, 0 to 9355
Data columns (total 18 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   timestamp                            9356 non-null   object 
 1   wind_speed_gfs                       125 non-null    float64
 2   wind_direction_gfs                   125 non-null    float64
 3   temp_alert_noaa                      337 non-null    float64
 4   temp_weekly_alert_noaa               337 non-null    float64
 5   dhw_noaa                             469 non-null    float64
 6   satellite_temperature_noaa           469 non-null    float64
 7   sst_anomaly_noaa                     469 non-null    float64
 8   top_temperature_spotter              8522 non-null   float64
 9   bottom_temperature_spotter           6993 non-null   float64
 10  significant_wave_height_spotter      8430 non-null   floa

The clean records for the Spotter buoy are obtained by applying the `get_clean_records` method of the `Buoy` class, which performs several processing steps to prepare the data for analysis. This includes standardizing column names to follow a consistent naming convention, converting all columns that contain "spotter" in their name to float data type.

In [46]:
dict_clean_spotter_sources['spotter_data_aqualink'].head()

,wind_speed_gfs,wind_direction_gfs,temp_alert_noaa,temp_weekly_alert_noaa,dhw_noaa,satellite_temperature_noaa,sst_anomaly_noaa,top_temp[C],bottom_temp[C],hs[m],tm[s],wave_mean_dir[°],wind_speed[m/s],wind_dir[°],significant_wave_height_sofar_model,wave_mean_period_sofar_model,wave_mean_direction_sofar_model
date,,,,,,,,,,,,,,,,,
2023-05-11 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.600000,28.580000,0.245,4.59,350.8575,0.8,11.0,NaN,NaN,NaN
2023-05-11 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.780000,28.613333,0.240,4.34,33.7290,1.0,39.5,NaN,NaN,NaN
2023-05-11 05:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.753333,28.680000,0.260,4.70,304.7405,0.8,36.5,NaN,NaN,NaN
2023-05-11 07:00:00,NaN,NaN,1.0,1.0,0.0,28.749942,1.093942,28.720000,28.726667,0.245,5.16,191.2130,0.4,337.0,NaN,NaN,NaN
2023-05-11 09:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.820000,28.820000,0.240,4.61,282.1440,1.0,262.5,NaN,NaN,NaN


In [47]:
dict_clean_spotter_sources['spotter_data_sofar'].head()

,Battery Voltage (V),Power (W),Humidity (%rel),Epoch Time,hs[m],tp[s],tm[s],wave_peak_dir[°],wave_peak_dir_spread[°],wave_mean_dir[°],...,Partition0 Mean Direction (deg),Partition0 Mean Directional Spread (deg),Partition1 Start Frequency (hz),Partition1 End Frequency (hz),Partition1 Significant Wave Height (m),Partition1 Mean Period (s),Partition1 Mean Direction (deg),Partition1 Mean Directional Spread (deg),Mean Barometric Pressure (hPa),Processing Source
date,,,,,,,,,,,,,,,,,,,,,
2025-05-11 01:10:00,4.01,-0.15,82.4,1746943800,0.649,7.877,3.301,139.800,73.297,67.257,...,-,-,-,-,-,-,-,-,1011.6,embedded
2025-05-11 03:10:00,4.01,-0.14,83.2,1746951000,0.631,7.314,3.343,153.838,70.252,69.970,...,-,-,-,-,-,-,-,-,1010.6,embedded
2025-05-11 05:10:00,4.01,-0.15,84.0,1746958200,0.590,8.533,3.142,47.366,73.935,63.592,...,-,-,-,-,-,-,-,-,1010.4,embedded
2025-05-11 07:10:00,4.01,0.07,84.0,1746965400,0.558,7.314,3.102,96.192,72.090,58.614,...,-,-,-,-,-,-,-,-,1010.7,embedded
2025-05-11 09:10:00,4.04,0.53,78.4,1746972600,0.546,8.533,3.060,164.476,78.778,65.990,...,-,-,-,-,-,-,-,-,1011.6,embedded


The column names can be inspected once the standardization is applied, and they follow a consistent naming convention that includes the variable name and the unit of measurement in square brackets.

In [48]:
for key in dict_clean_spotter_sources.keys():
    print(f"Clean records for {key}:")
    print(dict_clean_spotter_sources[key].info())

Clean records for spotter_data_aqualink:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 128 entries, 2023-05-11 01:00:00 to 2023-05-21 15:00:00
Data columns (total 17 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   wind_speed_gfs                       0 non-null      float64
 1   wind_direction_gfs                   0 non-null      float64
 2   temp_alert_noaa                      11 non-null     float64
 3   temp_weekly_alert_noaa               11 non-null     float64
 4   dhw_noaa                             11 non-null     float64
 5   satellite_temperature_noaa           11 non-null     float64
 6   sst_anomaly_noaa                     11 non-null     float64
 7   top_temp[C]                          118 non-null    float64
 8   bottom_temp[C]                       118 non-null    float64
 9   hs[m]                                128 non-null    float64
 10  tm[s]               

# HOBO sensors

In [49]:
obj_HOBO_temp = HOBO_Temp('../data/observations/HOBO/TL.csv',
                          start_dt=datetime(2025,3,16,0,0,0), 
                          end_dt=datetime(2025,3,18,0,0,0))

obj_HOBO_temp_cond = HOBO_TempCond('../data/observations/HOBO/CL.csv',
                            start_dt=datetime(2024,9,13,0,0,0), 
                            end_dt=datetime(2024,9,18,0,0,0))

In [50]:
dict_hobo_raw_measurements = dict()
dict_hobo_clean_measurements = dict()
for hobo in [obj_HOBO_temp, obj_HOBO_temp_cond]:
    raw_data = hobo.get_raw_records()
    clean_data = hobo.get_clean_records()
    dict_hobo_raw_measurements[hobo.__class__.__name__] = raw_data
    dict_hobo_clean_measurements[hobo.__class__.__name__] = clean_data

In [51]:
dict_hobo_raw_measurements['HOBO_Temp'].head()

,N.º,"Fecha Tiempo, GMT-05:00","Temp, °C (LGR S/N: 21907892, SEN S/N: 21907892)",Acoplador separado (LGR S/N: 21907892),Acoplador adjunto (LGR S/N: 21907892),Host conectado (LGR S/N: 21907892),Parado (LGR S/N: 21907892),Final de archivo (LGR S/N: 21907892)
0,1,03/15/25 11:04:00 AM,29.765,Registrado,NaN,NaN,NaN,NaN
1,2,03/15/25 11:09:00 AM,29.464,NaN,NaN,NaN,NaN,NaN
2,3,03/15/25 11:14:00 AM,29.590,NaN,NaN,NaN,NaN,NaN
3,4,03/15/25 11:19:00 AM,29.590,NaN,NaN,NaN,NaN,NaN
4,5,03/15/25 11:24:00 AM,29.265,NaN,NaN,NaN,NaN,NaN


In [52]:
dict_hobo_raw_measurements['HOBO_TempCond'].head()

,N.º,"Fecha Tiempo, GMT-05:00","Rango alto, μS/cm (LGR S/N: 22000823, SEN S/N: 22000823)","Temp, °C (LGR S/N: 22000823, SEN S/N: 22000823)",Acoplador separado (LGR S/N: 22000823),Acoplador adjunto (LGR S/N: 22000823),Host conectado (LGR S/N: 22000823),Parado (LGR S/N: 22000823),Final de archivo (LGR S/N: 22000823)
0,1,09/12/24 01:00:00 PM,252.0,33.88,Registrado,NaN,NaN,NaN,NaN
1,2,09/12/24 01:10:00 PM,413.6,34.40,NaN,NaN,NaN,NaN,NaN
2,3,09/12/24 01:20:00 PM,591.3,34.16,NaN,NaN,NaN,NaN,NaN
3,4,09/12/24 01:30:00 PM,628.3,33.76,NaN,NaN,NaN,NaN,NaN
4,5,09/12/24 01:40:00 PM,644.2,33.41,NaN,NaN,NaN,NaN,NaN


Some columns are not needed for the analysis, so they are dropped from the clean records.

In [53]:
for key in dict_hobo_raw_measurements.keys():
    print(f"\tRaw records for {key}:")
    print(dict_hobo_raw_measurements[key].info())

	Raw records for HOBO_Temp:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16156 entries, 0 to 16155
Data columns (total 8 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   N.º                                              16156 non-null  int64  
 1   Fecha Tiempo, GMT-05:00                          16156 non-null  object 
 2   Temp, °C (LGR S/N: 21907892, SEN S/N: 21907892)  16154 non-null  float64
 3   Acoplador separado (LGR S/N: 21907892)           1 non-null      object 
 4   Acoplador adjunto (LGR S/N: 21907892)            1 non-null      object 
 5   Host conectado (LGR S/N: 21907892)               1 non-null      object 
 6   Parado (LGR S/N: 21907892)                       1 non-null      object 
 7   Final de archivo (LGR S/N: 21907892)             1 non-null      object 
dtypes: float64(1), int64(1), object(6)
memory usage: 1009.9+ KB
None
	Raw records f

In [54]:
dict_hobo_clean_measurements['HOBO_Temp'].head()

,seq,temperature[C]
date,,
2025-03-16 00:04:00,157,28.072
2025-03-16 00:09:00,158,28.072
2025-03-16 00:14:00,159,28.048
2025-03-16 00:19:00,160,28.048
2025-03-16 00:24:00,161,28.048


In [55]:
dict_hobo_clean_measurements['HOBO_TempCond'].head()

,seq,temperature[C],conductivity[uS/cm]
date,,,
2024-09-13 00:00:00,67,30.86,23949.0
2024-09-13 00:10:00,68,30.87,23960.7
2024-09-13 00:20:00,69,30.86,23952.4
2024-09-13 00:30:00,70,30.85,23954.0
2024-09-13 00:40:00,71,30.85,23947.4


 The column names are standardized to follow a consistent naming convention that includes the variable name and the unit of measurement in square brackets. The date column is parsed using a specific format to ensure that it is correctly interpreted as a datetime object. Finally, the date column is set as the index of the DataFrame and sorted in chronological order.

In [56]:
for key in dict_hobo_clean_measurements.keys():
    print(f"\tClean records for {key}:")
    print(dict_hobo_clean_measurements[key].info())

	Clean records for HOBO_Temp:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 576 entries, 2025-03-16 00:04:00 to 2025-03-17 23:59:00
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   seq             576 non-null    int64  
 1   temperature[C]  576 non-null    float64
dtypes: float64(1), int64(1)
memory usage: 13.5 KB
None
	Clean records for HOBO_TempCond:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 721 entries, 2024-09-13 00:00:00 to 2024-09-18 00:00:00
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   seq                  721 non-null    int64  
 1   temperature[C]       721 non-null    float64
 2   conductivity[uS/cm]  721 non-null    float64
dtypes: float64(2), int64(1)
memory usage: 22.5 KB
None
